# 07.2 — Piecewise-linear schedules

Every curriculum lever — load augmentation, loss weights, freezing ([07.1](07.1_curriculum_learning_intuition.ipynb)) — rides on *one* mathematical primitive: `piecewise_anneal_value`, a piecewise-linear interpolator between waypoints. Give it a base value, a list of `(epoch, magnitude)` waypoints, and the current epoch, and it returns `base × magnitude(epoch)`, ramping linearly between waypoints. Simple — except it carries a deliberate **off-by-one quirk** from MATLAB that makes the ramp land one epoch "late" at every waypoint. This notebook is that primitive, and the interesting question of *why you'd port a bug on purpose*.

This notebook covers:

* Piecewise-linear interpolation: waypoints, ramps, and end-clamps.
* The `base × magnitude` structure — magnitudes are multipliers.
* The MATLAB `epoch − 1` off-by-one quirk and its per-waypoint discontinuity.
* Why parity means preserving the quirk (contrast with [06.8](../06_loss_orchestration/06.8_l2_inside_the_loss_kernel.ipynb)'s deliberate divergence).

**Prerequisites:** [07.1 curriculum intuition](07.1_curriculum_learning_intuition.ipynb).

## Section 1 — What MATLAB does

`cgg_calculateDynamicValue` finds the active segment and calls `cgg_annealWeight` to ramp within it:

```matlab
function w = cgg_annealWeight(minW, maxW, Epoch, DelayEpoch, RampLen)
    diff = maxW - minW;
    if Epoch <= DelayEpoch
        ramp = 0;
    elseif Epoch <= DelayEpoch + RampLen
        ramp = (Epoch - 1 - DelayEpoch) * (diff / RampLen);   % <- the (Epoch - 1)
    else
        ramp = diff;
    end
    w = minW + ramp;
end
```

That `Epoch - 1` is the quirk. It means the ramp uses the position `Epoch - 1 - DelayEpoch` instead of `Epoch - DelayEpoch`, so at the *end* of a segment the ramp reaches only `(RampLen − 1)/RampLen` of the way — and the value snaps to the true waypoint magnitude one epoch later. The port preserves this exactly (§2.4); the parity test pins the discontinuity at multiple waypoints.

## Section 2 — The Python concepts you need

### 2.1 — Piecewise-linear interpolation

A schedule is defined by **waypoints**: pairs of `(epoch, magnitude)`. Between consecutive waypoints the value ramps *linearly*; outside the waypoint range it *clamps* to the nearest end. So three waypoints give two ramps and two flat clamps (before the first, after the last):

In [ ]:
from neural_data_decoding.training.schedules.interpolator import piecewise_anneal_value

base = 1.0
epoch_points     = [3, 7]       # two waypoints
magnitude_points = [0.2, 1.0]   # ramp from 0.2× up to 1.0×
print("epoch | value")
for e in range(1, 11):
    print(f"  {e:2}  | {piecewise_anneal_value(base, epoch_points, magnitude_points, e):.3f}")
print("→ flat 0.2 before epoch 3, ramps to 1.0, flat 1.0 after epoch 7 (end-clamps).")

### 2.2 — `base × magnitude`: magnitudes are multipliers

The magnitudes aren't the values — they're *multipliers on a static base*. `piecewise_anneal_value(base, ..., mag, epoch)` returns `base × magnitude(epoch)`. This lets one schedule shape (waypoints + magnitudes) apply to *any* base value: a KL weight of base 0.5 and a base 2.0 use the *same* `[0.0, 1.0]` ramp, scaled differently. Separating the *shape* (magnitudes) from the *scale* (base) is what makes the schedules reusable across parameters.

In [ ]:
for base in (0.5, 2.0):
    vals = [round(piecewise_anneal_value(base, [3, 7], [0.0, 1.0], e), 2) for e in range(1, 10)]
    print(f"base={base}: {vals}")
print("→ same [0.0, 1.0] ramp SHAPE, scaled by each base — magnitudes are reusable multipliers.")

### 2.3 — The off-by-one quirk: the ramp lands late

In [ ]:
# Waypoints [2, 6, 10] with magnitudes [0, 1, 2]. At each INTERNAL waypoint,
# the value is NOT the waypoint's magnitude — it's (span-1)/span of the way.
wp, mg = [2, 6, 10], [0.0, 1.0, 2.0]
print("epoch | value  | at a waypoint?")
for e in range(1, 13):
    v = piecewise_anneal_value(1.0, wp, mg, e)
    tag = f"← waypoint magnitude={mg[wp.index(e)]}, but value={v:.2f}" if e in wp else ""
    print(f"  {e:2}  | {v:.3f}  {tag}")
print()
print("At epoch 6 (waypoint, magnitude 1.0) the value is 0.75 = (4-1)/4 of the ramp.")
print("It only reaches 1.0 at epoch 7 — one epoch LATE. That's the MATLAB quirk.")

Look at epoch 6: it's a waypoint with magnitude `1.0`, but the value is `0.75`. The segment `[2, 6]` has span 4, and the `Epoch − 1` position makes the ramp reach only `(4−1)/4 = 3/4` of the way by the segment's *last* epoch. The value snaps to the true `1.0` at epoch 7 — the leading edge of the next segment. Every internal waypoint has this one-epoch lag and the little discontinuity that comes with it.

### 2.4 — Draw the quirk against the "ideal" ramp

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

epochs = list(range(1, 13))
actual = [piecewise_anneal_value(1.0, wp, mg, e) for e in epochs]
# "Ideal" textbook piecewise-linear (numpy.interp, no off-by-one):
ideal = [float(np.interp(e, wp, mg)) for e in epochs]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs, ideal, "s--", color="gray", label="ideal piecewise-linear (np.interp)")
ax.plot(epochs, actual, "o-", color="crimson", label="actual (MATLAB quirk)")
for x in wp:
    ax.axvline(x, color="black", ls=":", lw=0.7)
ax.scatter(wp, mg, color="black", zorder=5, s=60, label="waypoints (epoch, magnitude)")
ax.set_xlabel("epoch"); ax.set_ylabel("scheduled value"); ax.set_title("The off-by-one: actual ramp reaches each waypoint one epoch late")
ax.legend()
plt.tight_layout(); plt.show()
print("Gray hits each waypoint on time; red lags by one epoch at every internal waypoint.")

The gray "textbook" curve passes exactly through each waypoint; the red "actual" curve lags — it reaches each waypoint's magnitude one epoch after the waypoint. That gap is not a rounding error; it's a systematic consequence of MATLAB's `Epoch − 1`.

### 2.5 — Why port a quirk on purpose?

This is the mirror image of [06.8](../06_loss_orchestration/06.8_l2_inside_the_loss_kernel.ipynb), where the port *deliberately diverged* from MATLAB (AdamW instead of Adam + grad-side L2). Here the port *deliberately matches* — quirk and all. Why the difference?

* In 06.8, MATLAB's Adam-L2 was a known-*suboptimal* recipe the field has improved on; reproducing it would bake in a defect.
* Here, the off-by-one is *cosmetic* — a one-epoch lag in a schedule that runs for dozens of epochs. It changes nothing about the training's correctness, but it *does* change the exact per-epoch numbers. To validate the port against MATLAB fixtures ([06.10](../06_loss_orchestration/06.10_nan_masked_reconstruction.ipynb)), the numbers must match *bit-for-bit* — so the quirk is preserved and pinned by a parity test.

The rule: match the reference by default (so parity tests pass); diverge only when the reference is doing something *wrong* that would harm results. A harmless cosmetic quirk is matched; a genuine defect is fixed. Knowing which is which requires understanding *why* the reference does what it does — not just copying or "improving" blindly.

## Section 3 — The `neural_data_decoding` implementation

`piecewise_anneal_value` — the clamps, the segment search, and the inlined `cgg_annealWeight`:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/schedules/interpolator.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "if epoch <= epoch_points[0]:" in line)
for k in range(i, i + 30):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* The two **end-clamps** (§2.1): `epoch <= epoch_points[0]` → first magnitude; `epoch > epoch_points[-1]` → last magnitude.
* The **segment search** iterates from the end to find the largest `idx` with `epoch_points[idx] < epoch` — the comment notes this mirrors MATLAB's `find(EpochPoints < Epoch, 1, 'last')`.
* The inlined `cgg_annealWeight`: `ramp = (epoch - 1 - delay) * (target_diff / ramp_len)` — the **`epoch - 1`** off-by-one (§2.3), preserved with a comment saying so.
* The `ramp_len <= 0 or isnan(ramp_len)` guard handles degenerate/zero-length segments (returns the full `target_diff`), and `isnan(base)` short-circuits to NaN — a NaN base means "parameter inactive" ([07.4](07.4_loss_weights_curriculum.ipynb)).
* Everything is 1-indexed (epochs start at 1, matching MATLAB) — a common off-by-one source when porting ([02.1](../02_numpy_and_pytorch_basics/02.1_numpy_vs_matlab_arrays.ipynb) territory), handled consistently here.

## Section 4 — Hands-on exercises

### Exercise 4.1 — predict the clamps

Without running it, what does a schedule with waypoints `[4, 8]`, magnitudes `[0.5, 2.0]`, base `10` return at epoch 1, epoch 100? Then check.

In [ ]:
# Reveal:
print("epoch 1 (before first waypoint):", piecewise_anneal_value(10.0, [4, 8], [0.5, 2.0], 1),
      "= base×first magnitude = 10×0.5")
print("epoch 100 (after last waypoint):", piecewise_anneal_value(10.0, [4, 8], [0.5, 2.0], 100),
      "= base×last magnitude = 10×2.0")
print("→ outside the waypoint range, the value clamps flat to the nearest end.")

### Exercise 4.2 — measure the discontinuity

For waypoints `[1, 5]`, magnitudes `[0, 1]`, compute the jump between epoch 5 (segment end) and epoch 6 (right-clamp). Show it equals `1/span`.

In [ ]:
# Reveal:
v5 = piecewise_anneal_value(1.0, [1, 5], [0.0, 1.0], 5)
v6 = piecewise_anneal_value(1.0, [1, 5], [0.0, 1.0], 6)
span = 5 - 1
print(f"epoch 5 (segment end): {v5:.3f}   epoch 6 (clamp): {v6:.3f}")
print(f"jump = {v6 - v5:.3f}   vs   1/span = 1/{span} = {1/span:.3f}")
print("→ the off-by-one leaves exactly 1/span of the ramp to be closed by the snap.")

## Section 5 — Common errors

### The scheduled value is one epoch behind what you expect

That's the off-by-one quirk (§2.3), not a bug in your config. The ramp reaches each waypoint's magnitude one epoch late. If you need a value hit *exactly* at epoch N, place the waypoint at N−1 to compensate — but for parity with MATLAB, leave it.

### Waypoints not monotonically increasing

`epoch_points` must ascend. The segment search assumes it; unsorted points pick the wrong segment. (The library builders enforce this — [07.3](07.3_load_parameters.ipynb).)

### `epoch_points` and `magnitude_points` different lengths

Raises `ValueError` — every waypoint epoch needs a magnitude. A common slip when editing one list but not the other.

### A NaN sneaks into the schedule

`isnan(base)` returns NaN deliberately (an inactive parameter). But a NaN in `magnitude_points` propagates through the ramp — check your magnitudes are finite unless you mean "inactive."

### Expecting `np.interp` behavior

`numpy.interp` is the *textbook* piecewise-linear and does **not** have the off-by-one (§2.4). If you validate the port against `np.interp`, it'll mismatch at every waypoint — validate against MATLAB fixtures, not against the "ideal.

## Section 6 — Further reading

- [`src/neural_data_decoding/training/schedules/interpolator.py`](../../src/neural_data_decoding/training/schedules/interpolator.py) — the full primitive with the parity docstring.
- [Linear interpolation (Wikipedia)](https://en.wikipedia.org/wiki/Linear_interpolation) — the textbook version the quirk deviates from.
- [06.8 L2 inside the loss kernel](../06_loss_orchestration/06.8_l2_inside_the_loss_kernel.ipynb) — the contrasting case where the port *diverges* from MATLAB.

Next up: **[07.3 load parameters](07.3_load_parameters.ipynb)** — the first curriculum lever: scheduling the data-loading/augmentation knobs.